In [32]:
# Пример данных
products_data = [
    (1, 'Apple'),
    (2, 'Banana'),
    (3, 'Carrot'),
    (4, 'Beef Steak'),
    (5, 'Chicken Breast'),
    (6, 'Milk'),
    (7, 'Cheese'),
    (8, 'Bread'),
    (9, 'Shampoo'),
    (10, 'Toothpaste')
]

categories_data = [
    (1, 'Fruits'),
    (2, 'Vegetables'),
    (3, 'Meat'),
    (4, 'Dairy'),
    (5, 'Bakery')
]

product_categories_data = [
    (1, 1),  # Apple -> Fruits
    (2, 1),  # Banana -> Fruits
    (3, 2),  # Carrot -> Vegetables
    (4, 3),  # Beef Steak -> Meat
    (5, 3),  # Chicken Breast -> Meat
    (6, 4),  # Milk -> Dairy
    (7, 4),  # Cheese -> Dairy
    (8, 5)   # Bread -> Bakery
]

In [33]:
spark = SparkSession.builder.appName("ProductAndCategories").getOrCreate()

# Создание датафреймов
products_df = spark.createDataFrame(products_data, ["product_id", "product_name"])
categories_df = spark.createDataFrame(categories_data, ["category_id", "category_name"])
product_categories_df = spark.createDataFrame(product_categories_data, ["product_id", "category_id"])

In [34]:
def get_product_category_pairs_and_unlinked_products(products_df, categories_df, product_categories_df):
    # Соединяем продукты с категориями
    joined_df = products_df.join(product_categories_df, on="product_id", how="left")\
                           .join(categories_df, on="category_id", how="left")
    print(joined_df.show())
    # Получаем все пары "Имя продукта – Имя категории"
    product_category_pairs_df = joined_df.select("product_name", "category_name")\
                                         .filter(col("category_name").isNotNull())
    
    # Получаем продукты, у которых нет категорий
    unlinked_products_df = joined_df.filter(col("category_name").isNull())\
                                    .select("product_name").distinct()
    
    return product_category_pairs_df, unlinked_products_df

In [35]:
product_category_pairs_df, unlinked_products_df = get_product_category_pairs_and_unlinked_products(products_df, categories_df, product_categories_df)

print("Product-Category Pairs:")
product_category_pairs_df.show()

print("Products with no categories:")
unlinked_products_df.show()

+-----------+----------+--------------+-------------+
|category_id|product_id|  product_name|category_name|
+-----------+----------+--------------+-------------+
|       NULL|         9|       Shampoo|         NULL|
|       NULL|        10|    Toothpaste|         NULL|
|          5|         8|         Bread|       Bakery|
|          1|         1|         Apple|       Fruits|
|          1|         2|        Banana|       Fruits|
|          3|         5|Chicken Breast|         Meat|
|          3|         4|    Beef Steak|         Meat|
|          2|         3|        Carrot|   Vegetables|
|          4|         6|          Milk|        Dairy|
|          4|         7|        Cheese|        Dairy|
+-----------+----------+--------------+-------------+

None
Product-Category Pairs:
+--------------+-------------+
|  product_name|category_name|
+--------------+-------------+
|        Banana|       Fruits|
|         Apple|       Fruits|
|        Carrot|   Vegetables|
|Chicken Breast|         Mea